In [2]:
import pandas as pd
import numpy as np

In [2]:
# Load all 10 provided CSV datasets using Pandas.
# Print .shape, .dtypes, and .head() for each. Note any anomalies.

In [30]:
df_fund_master = pd.read_csv('../data/raw/01_fund_master.csv')
df_02_nav_history = pd.read_csv('../data/raw/02_nav_history.csv')
df_03_aum_by_fund_house = pd.read_csv('../data/raw/03_aum_by_fund_house.csv')
df_04_monthly_sip_inflows = pd.read_csv('../data/raw/04_monthly_sip_inflows.csv')
df_05_category_inflows = pd.read_csv('../data/raw/05_category_inflows.csv')
df_06_industry_folio_count = pd.read_csv('../data/raw/06_industry_folio_count.csv')
df_07_scheme_performance = pd.read_csv('../data/raw/07_scheme_performance.csv')
df_08_investor_transactions = pd.read_csv('../data/raw/08_investor_transactions.csv')
df_09_portfolio_holdings = pd.read_csv('../data/raw/09_portfolio_holdings.csv')
df_10_benchmark_indices = pd.read_csv('../data/raw/10_benchmark_indices.csv')

## DATA CLEANING

### 1 Nav history

In [4]:
df_02_nav_history.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [5]:
df_02_nav_history.dtypes

amfi_code      int64
date          object
nav          float64
dtype: object

In [6]:
#parse dates into datetime
df_02_nav_history['date'] = pd.to_datetime(df_02_nav_history['date'], format='mixed', errors='coerce')

In [7]:
df_02_nav_history.dtypes

amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object

In [8]:
df_02_nav_history.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


In [12]:
from pandas.api.types import (
    is_numeric_dtype,
    is_integer_dtype,
    is_float_dtype,
    is_bool_dtype,
    is_datetime64_any_dtype,
    is_object_dtype,
    is_string_dtype,
    is_categorical_dtype
)

In [13]:
print(is_datetime64_any_dtype(df_02_nav_history['date']))   # True

True


In [15]:
#sort by amfi_code+date
df_02_nav_history = df_02_nav_history.sort_values(by=['amfi_code','date'])

In [16]:
df_02_nav_history

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


In [31]:
df_02_nav_history.isna().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [36]:
df_02_nav_history.duplicated().sum()

np.int64(0)

In [ ]:
#drop duplicates
df_02_nav_history=df_02_nav_history.drop_duplicates()

In [37]:
#Validate nav>0
df_02_nav_history = df_02_nav_history[df_02_nav_history['nav']>0]

,amfi_code,date,nav


###### forword filling

In [20]:
#forward fill for missing nav

start_date = df_02_nav_history['date'].min()
end_date   = df_02_nav_history['date'].max()
full_dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Function to forward-fill one scheme
def forward_fill_scheme(group):
    group = group.set_index('date')
    group = group.reindex(full_dates)
    group['nav'] = group['nav'].ffill()
    group['amfi_code'] = group['amfi_code'].ffill()
    group = group.reset_index().rename(columns={'index': 'date'})
    return group

# Apply per amfi_code
df_02_nav_history = df_02_nav_history.groupby('amfi_code', group_keys=False).apply(forward_fill_scheme)

# Restore original column order if needed
df_02_nav_history = df_02_nav_history[['amfi_code', 'date', 'nav'] + 
                      [col for col in df_02_nav_history.columns if col not in ['amfi_code', 'date', 'nav']]]

print(df_02_nav_history.head())


   amfi_code       date       nav
0   100016.0 2022-01-03  520.4608
1   100016.0 2022-01-04  515.0971
2   100016.0 2022-01-05  521.7239
3   100016.0 2022-01-06  515.7880
4   100016.0 2022-01-07  515.1639


C:\Users\Ashwin\AppData\Local\Temp\ipykernel_34100\725759379.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_02_nav_history = df_02_nav_history.groupby('amfi_code', group_keys=False).apply(forward_fill_scheme)


In [38]:
#exporting the cleaned file
df_02_nav_history.to_csv('../data/processed/nav_history_cleaned.csv', index=False)

### 2 investor transaction cleaning 

In [22]:
df_08_investor_transactions.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [31]:
status_mapping = {
    'SIP':['SIP','sip', 'systematic investment', 'sys_inv', 'monthly_sip', 's.i.p.'],
    'Lumpsum':['Lumpsum','lumpsum', 'lump sum', 'one-time', 'onetime', 'one time', 'purchase'],
    'Redemption': ['Redemption','redemption', 'redeem', 'sell', 'withdrawal', 'payout']
}
flat_mapping= {varient:standerd for standerd, varients in status_mapping.items() for varient in varients}
df_08_investor_transactions['transaction_type'] = df_08_investor_transactions['transaction_type'].map(flat_mapping).fillna('Other').str.capitalize()

In [33]:
#Standardize transaction type
df_08_investor_transactions.transaction_type=df_08_investor_transactions.transaction_type.str.upper().str.strip()

In [34]:
df_08_investor_transactions['transaction_type'].unique()

array(['SIP', 'REDEMPTION', 'LUMPSUM'], dtype=object)

In [37]:
#Validate amount
df_08_investor_transactions=df_08_investor_transactions[df_08_investor_transactions.amount_inr>0]

In [38]:
df_08_investor_transactions[df_08_investor_transactions['amount_inr']<0]

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status


In [40]:
#Validate KYC
valid_kyc=['Verified','Pending','Rejected']
df_08_investor_transactions=df_08_investor_transactions[df_08_investor_transactions.kyc_status.isin(valid_kyc)]

In [41]:
df_08_investor_transactions['kyc_status'].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [42]:
df_08_investor_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [45]:
df_08_investor_transactions=df_08_investor_transactions.rename(columns={'transaction_date':'date'})

In [46]:
df_08_investor_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   date                32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [47]:
df_08_investor_transactions['date'] = pd.to_datetime(df_08_investor_transactions['date'], format='mixed', errors='coerce')

In [48]:
df_08_investor_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   date                32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [49]:
df_08_investor_transactions.to_csv('../data/processed/08_investor_transactions_cleaned.csv', index=False)

### 3 scheme performance cleaning

In [32]:
df_07_scheme_performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [33]:
df_07_scheme_performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [50]:
#Validate returns to numeric
returns=['return_1yr_pct','return_3yr_pct','return_5yr_pct']

for col in returns:
  df_07_scheme_performance[col]=pd.to_numeric(df_07_scheme_performance[col],errors='coerce')

In [51]:
df_07_scheme_performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [52]:
#Flag anomalies
df_07_scheme_performance['anomaly']=df_07_scheme_performance[returns].isnull().any(axis=1)

In [53]:
df_07_scheme_performance[(df_07_scheme_performance['expense_ratio_pct']>.1) | (df_07_scheme_performance['expense_ratio_pct']<2.5)]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,anomaly
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,False
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,False
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,False
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,False
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,False
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate,False
6,125497,HDFC Top 100 Fund - Direct Plan - Growth,HDFC Mutual Fund,Large Cap,Direct,11.48,13.38,13.48,12.25,1.13,0.97,0.96,1.45,14.0,-33.50,10611,0.92,4,Moderate,False
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,15.43,16.58,17.69,15.63,0.95,0.91,0.87,1.44,19.0,-13.67,23185,1.38,5,High,False
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High,False
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low,False


In [35]:
df_07_scheme_performance[(df_07_scheme_performance['expense_ratio_pct']>.1) | (df_07_scheme_performance['expense_ratio_pct']<2.5)]['amfi_code'].count()

np.int64(40)

In [36]:
df_07_scheme_performance[(df_07_scheme_performance['expense_ratio_pct']<.1) | (df_07_scheme_performance['expense_ratio_pct']>2.5)]['amfi_code'].count()

np.int64(0)

In [37]:
df_07_scheme_performance.to_csv('../data/processed/scheme_performance_cleaned.csv', index=False)

## Cleaning Remaning datasets

In [4]:

dt_1=pd.read_csv('../data/raw/01_fund_master.csv')
dt_3=pd.read_csv('../data/raw/03_aum_by_fund_house.csv')
dt_4=pd.read_csv('../data/raw/04_monthly_sip_inflows.csv')
dt_5=pd.read_csv('../data/raw/05_category_inflows.csv')
dt_6=pd.read_csv('../data/raw/06_industry_folio_count.csv')
dt_9=pd.read_csv('../data/raw/09_portfolio_holdings.csv')
dt_10=pd.read_csv('../data/raw/10_benchmark_indices.csv')

In [5]:
dt_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   fund_house          40 non-null     object 
 2   scheme_name         40 non-null     object 
 3   category            40 non-null     object 
 4   sub_category        40 non-null     object 
 5   plan                40 non-null     object 
 6   launch_date         40 non-null     object 
 7   benchmark           40 non-null     object 
 8   expense_ratio_pct   40 non-null     float64
 9   exit_load_pct       40 non-null     float64
 10  min_sip_amount      40 non-null     int64  
 11  min_lumpsum_amount  40 non-null     int64  
 12  fund_manager        40 non-null     object 
 13  risk_category       40 non-null     object 
 14  sebi_category_code  40 non-null     object 
dtypes: float64(2), int64(3), object(10)
memory usage: 4.8+ KB


In [6]:
dt_1.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,14-02-2006,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,01-01-2013,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,09-09-2009,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,01-01-2013,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,30-12-2000,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [7]:
dt_1=dt_1.rename(columns={'launch_date':'date'})

In [8]:
dt_1.date=pd.to_datetime(dt_1.date,errors='coerce')
dt_1.drop_duplicates()
dt_1.isna().sum()

C:\Users\Ashwin\AppData\Local\Temp\ipykernel_22392\1791522548.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dt_1.date=pd.to_datetime(dt_1.date,errors='coerce')


amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
date                  0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

In [9]:
dt_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   amfi_code           40 non-null     int64         
 1   fund_house          40 non-null     object        
 2   scheme_name         40 non-null     object        
 3   category            40 non-null     object        
 4   sub_category        40 non-null     object        
 5   plan                40 non-null     object        
 6   date                40 non-null     datetime64[ns]
 7   benchmark           40 non-null     object        
 8   expense_ratio_pct   40 non-null     float64       
 9   exit_load_pct       40 non-null     float64       
 10  min_sip_amount      40 non-null     int64         
 11  min_lumpsum_amount  40 non-null     int64         
 12  fund_manager        40 non-null     object        
 13  risk_category       40 non-null     object        
 

##### df3

In [10]:
dt_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            90 non-null     object 
 1   fund_house      90 non-null     object 
 2   aum_lakh_crore  90 non-null     float64
 3   aum_crore       90 non-null     int64  
 4   num_schemes     90 non-null     int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 3.6+ KB


In [11]:
dt_3.date=pd.to_datetime(dt_3.date,errors='coerce')
dt_3.drop_duplicates()
dt_3.isna().sum()

date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

In [12]:
dt_4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   month                      48 non-null     object 
 1   sip_inflow_crore           48 non-null     int64  
 2   active_sip_accounts_crore  48 non-null     float64
 3   new_sip_accounts_lakh      48 non-null     float64
 4   sip_aum_lakh_crore         48 non-null     float64
 5   yoy_growth_pct             36 non-null     float64
dtypes: float64(4), int64(1), object(1)
memory usage: 2.4+ KB


In [13]:
dt_4=dt_4.rename(columns={'month':'date'})

In [14]:
dt_4.date=pd.to_datetime(dt_4.date,errors='coerce')
dt_4.drop_duplicates()
dt_4.isna().sum()

date                          0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

In [15]:
dt_4=dt_4.dropna()

In [16]:
dt_4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 12 to 47
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   date                       36 non-null     datetime64[ns]
 1   sip_inflow_crore           36 non-null     int64         
 2   active_sip_accounts_crore  36 non-null     float64       
 3   new_sip_accounts_lakh      36 non-null     float64       
 4   sip_aum_lakh_crore         36 non-null     float64       
 5   yoy_growth_pct             36 non-null     float64       
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 2.0 KB


In [17]:
dt_4.head()

,date,sip_inflow_crore,active_sip_accounts_crore,new_sip_accounts_lakh,sip_aum_lakh_crore,yoy_growth_pct
12,2023-01-01,13856,6.13,9.10,5.82,20.31
13,2023-02-01,13687,6.19,8.20,5.88,19.66
14,2023-03-01,14276,6.32,10.00,6.01,15.80
15,2023-04-01,14749,6.41,9.25,6.15,24.33
16,2023-05-01,14749,6.50,8.70,6.22,20.05


##### df5

In [18]:
dt_5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   month             144 non-null    object 
 1   category          144 non-null    object 
 2   net_inflow_crore  144 non-null    float64
dtypes: float64(1), object(2)
memory usage: 3.5+ KB


In [19]:
dt_5=dt_5.rename(columns={'month':'date'})

In [20]:
dt_5.date=pd.to_datetime(dt_5.date,errors='coerce')
dt_5.drop_duplicates()
dt_5.isna().sum()
     


date                0
category            0
net_inflow_crore    0
dtype: int64

##### df6


In [21]:
dt_6.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                21 non-null     object 
 1   total_folios_crore   21 non-null     float64
 2   equity_folios_crore  21 non-null     float64
 3   debt_folios_crore    21 non-null     float64
 4   hybrid_folios_crore  21 non-null     float64
 5   others_folios_crore  21 non-null     float64
dtypes: float64(5), object(1)
memory usage: 1.1+ KB


In [22]:
dt_6=dt_6.rename(columns={'month':'date'})

In [23]:
dt_6.date=pd.to_datetime(dt_6.date,errors='coerce')
dt_6.drop_duplicates()
dt_6.isna().sum()
     

date                   0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      0
hybrid_folios_crore    0
others_folios_crore    0
dtype: int64

##### df9

In [24]:
dt_9.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   amfi_code          322 non-null    int64  
 1   stock_symbol       322 non-null    object 
 2   stock_name         322 non-null    object 
 3   sector             322 non-null    object 
 4   weight_pct         322 non-null    float64
 5   market_value_cr    322 non-null    float64
 6   current_price_inr  322 non-null    float64
 7   portfolio_date     322 non-null    object 
dtypes: float64(3), int64(1), object(4)
memory usage: 20.3+ KB


In [25]:
dt_9=dt_9.rename(columns={'portfolio_date':'date'})

In [26]:
dt_9.date=pd.to_datetime(dt_9.date,errors='coerce')
dt_9.drop_duplicates()
dt_9.isna().sum()

amfi_code            0
stock_symbol         0
stock_name           0
sector               0
weight_pct           0
market_value_cr      0
current_price_inr    0
date                 0
dtype: int64

##### df0


In [27]:
dt_10.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8050 entries, 0 to 8049
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         8050 non-null   object 
 1   index_name   8050 non-null   object 
 2   close_value  8050 non-null   float64
dtypes: float64(1), object(2)
memory usage: 188.8+ KB


In [28]:
dt_10.date=pd.to_datetime(dt_10.date,errors='coerce')
dt_10.drop_duplicates()
dt_10.isna().sum()

date           0
index_name     0
close_value    0
dtype: int64

In [29]:
dt_1.to_csv('../data/processed/01_fund_master_cleaned.csv',index=False)
dt_3.to_csv('../data/processed/03_aum_by_fund_house_cleaned.csv',index=False)
dt_4.to_csv('../data/processed/04_monthly_sip_inflows_cleaned.csv',index=False)
dt_5.to_csv('../data/processed/05_category_inflows_cleaned.csv',index=False)
dt_6.to_csv('../data/processed/06_industry_folio_count_cleaned.csv',index=False)
dt_9.to_csv('../data/processed/09_portfolio_holdings_cleaned.csv',index=False)
dt_10.to_csv('../data/processed/10_benchmark_indices_cleaned.csv',index=False)